# Chest X-Ray Pneumonia Detection — Google Colab

Train on **free cloud GPU** (T4 recommended).

**Before running:**
1. **Runtime → Change runtime type → GPU** (T4 or better)
2. Have a [Kaggle](https://www.kaggle.com) account and accept the [dataset rules](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)
3. Get your API token: Kaggle → Account → **Create New Token** → downloads `kaggle.json`

> ⚠️ Research/education only — not for clinical use.

## 1. Enable GPU

Menu: **Runtime → Change runtime type → Hardware accelerator: GPU**

In [1]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ No GPU detected. Go to Runtime → Change runtime type → GPU')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Project paths (Google Drive)

Your folder: [chest_xray_pneumonia_detection on Drive](https://drive.google.com/drive/folders/1--_EhLNau9vcpeH8gv1XxBD_brfMYQ16)

**Important:** The link uses `/drive/u/1/` = a **second Google account**. Colab must sign in with **that same account** (Cmd+Shift+P → `Colab: Sign Out` → reconnect → Mount Drive).

**If `drive.mount` fails in Cursor:** Cmd+Shift+P → `Colab: Mount Google Drive to Server`, then re-run the cell below.

In [5]:
import os
import sys
from pathlib import Path

# Your Drive folder (has src/, data/, notebook/, ...)
DRIVE_FOLDER = "chest_xray_pneumonia_detection"
DRIVE_FOLDER_ID = "1--_EhLNau9vcpeH8gv1XxBD_brfMYQ16"  # from your Drive URL

def mount_drive_safe():
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.exists():
        try:
            next(mydrive.iterdir())
            print("Google Drive already mounted.")
            return True
        except StopIteration:
            pass
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        print("Google Drive mounted.")
        return True
    except Exception as e:
        print("drive.mount() failed — use Colab extension command instead:")
        print("  Cmd+Shift+P → Colab: Mount Google Drive to Server")
        print(f"  ({e})")
        return False

mount_drive_safe()

def _has_src(p):
    return (p / "src" / "train.py").exists()

def _search_mydrive(max_depth=5):
    """Find src/train.py anywhere under My Drive."""
    root = Path("/content/drive/MyDrive")
    skip = {".git", ".venv", "chest_xray", "train", "test", "val", "__pycache__", "NORMAL", "PNEUMONIA"}
    queue = [(root, 0)]
    while queue:
        folder, depth = queue.pop(0)
        if _has_src(folder):
            return folder.resolve()
        if depth >= max_depth:
            continue
        try:
            kids = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".") and c.name not in skip]
        except (OSError, PermissionError):
            continue
        for c in sorted(kids):
            queue.append((c, depth + 1))
    return None

def find_project_root():
    here = Path.cwd()
    # Google Drive shortcut by folder ID (works when folder is on another account's shared mount)
    shortcut = Path(
        f"/content/drive/MyDrive/.shortcut-targets-by-id/by-id/{DRIVE_FOLDER_ID}"
    )
    candidates = [
        shortcut,
        Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}"),
        Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}/{DRIVE_FOLDER}"),
        Path("/content/drive/MyDrive/Chext_X Pneumonia detection/chest_xray_pneumonia_detection"),
        Path("/content/drive/MyDrive/OTU WINTER 2026/Project 3/Chext_X Pneumonia detection/chest_xray_pneumonia_detection"),
        here.parent if here.name == "notebook" else here,
        here,
    ]
    for p in candidates:
        if _has_src(p):
            return p.resolve()
    found = _search_mydrive()
    if found:
        print("Auto-found project:", found)
        return found
    return candidates[0]

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "chest_xray"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
BATCH_SIZE = 16

if not _has_src(PROJECT_ROOT):
    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("\nFolder contents (your Drive upload may be incomplete):")
    p = PROJECT_ROOT
    if p.exists():
        for x in sorted(p.iterdir())[:20]:
            print(" ", x.name + ("/" if x.is_dir() else ""))
    else:
        print("  (folder missing)")
    print("\nMy Drive top-level folders:")
    for x in sorted(Path("/content/drive/MyDrive").iterdir())[:15]:
        print(" ", x.name + ("/" if x.is_dir() else ""))
    print("\nSearching entire Drive mount for src/train.py ...")
    !find /content/drive -path "*/src/train.py" 2>/dev/null | head -10
    raise FileNotFoundError(
        "src/ not found on the mounted Drive account.\n"
        "Your files are here (account u/1): drive.google.com/.../1--_EhLNau9vcpeH8gv1XxBD_brfMYQ16\n"
        "Fix: Cmd+Shift+P → Colab: Sign Out → reconnect with THAT Google account → Mount Drive → re-run."
    )

for d in (FIGURES_DIR, MODELS_DIR, DATA_ROOT.parent):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src/ exists:", (PROJECT_ROOT / "src").exists())

Google Drive already mounted.
PROJECT_ROOT: /content/drive/MyDrive/chest_xray_pneumonia_detection
src/ exists: False


FileNotFoundError: Project not found. Mount Drive (see above), then ensure folder exists:
  My Drive/chest_xray_pneumonia_detection/src/

## 3. Fix missing `src/` (only if cell 2 failed)

If Drive folder is empty or only has the notebook, **zip & upload** the full project here:

### Option A — Clone from GitHub (if you pushed the repo)

In [ ]:
# Run ONLY if cell 2 said src/ missing
# On Mac: right-click chest_xray_pneumonia_detection → Compress → upload the .zip here

UPLOAD_PROJECT_ZIP = True  # set False after first successful upload

if UPLOAD_PROJECT_ZIP:
    from google.colab import files
    import zipfile

    print("Select chest_xray_pneumonia_detection.zip from your Mac...")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    dest = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}")
    dest.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(dest.parent)
    print("Extracted to Drive. Re-run cell 2 (paths).")
else:
    print("Skipped upload.")

### Option B — Upload project zip (easiest if not on GitHub)

On your Mac, zip the `chest_xray_pneumonia_detection` folder, then run the cell below and upload.

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path

UPLOAD = False  # Set True first time to upload zip

if UPLOAD:
    uploaded = files.upload()  # Select chest_xray_pneumonia_detection.zip
    zip_name = list(uploaded.keys())[0]
    !unzip -q "{zip_name}" -d /content/
    !cp -r /content/chest_xray_pneumonia_detection "{DRIVE_PROJECT}"
    print('Copied project to Google Drive.')

PROJECT_ROOT = Path(DRIVE_PROJECT)
if not (PROJECT_ROOT / 'src').exists():
  PROJECT_ROOT = Path('/content/chest_xray_pneumonia_detection')  # fallback if unzipped to /content

print(f'Project root: {PROJECT_ROOT}')

### Option C — Copy from Drive (if you already uploaded the folder)

In [3]:
# Already configured in cell 2 — confirm paths
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
assert (PROJECT_ROOT / "src").exists(), "Run cell 2 first and mount Drive"

NameError: name 'DRIVE_PROJECT' is not defined

## 4. Install dependencies

In [ ]:
%cd {PROJECT_ROOT}
!pip install -q torch torchvision timm grad-cam scikit-learn matplotlib seaborn tqdm Pillow kaggle

## 5. Kaggle API setup

Upload `kaggle.json` when prompted (one-time per session), **or** use Colab Secrets (recommended).

In [ ]:
import json
import os
from pathlib import Path
from google.colab import files

os.makedirs('/root/.kaggle', exist_ok=True)

USE_SECRETS = False  # Set True if you added KAGGLE_USERNAME + KAGGLE_KEY in Colab Secrets

if USE_SECRETS:
    from google.colab import userdata
    creds = {
        'username': userdata.get('KAGGLE_USERNAME'),
        'key': userdata.get('KAGGLE_KEY'),
    }
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(creds, f)
else:
    print('Upload kaggle.json:')
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle credentials ready.')

## 6. Download dataset (saved to Drive — only runs once)

In [ ]:
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'chest_xray'

if (DATA_ROOT / 'train').exists():
    print(f'Dataset already exists at {DATA_ROOT}')
else:
    print('Downloading from Kaggle (~1.2 GB)...')
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p {PROJECT_ROOT / 'data' / 'raw'} --unzip
    print('Done.')

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import get_dataset_stats
import json
print(json.dumps(get_dataset_stats(DATA_ROOT), indent=2))

## 7. Train on GPU

~20–40 min on Colab T4 for 15 epochs.

In [ ]:
from src.train import train

history = train(
    data_root=DATA_ROOT,
    backbone='resnet50',
    epochs=15,
    batch_size=32,      # Reduce to 16 if you get CUDA OOM
    lr=1e-4,
    output_dir=PROJECT_ROOT / 'models',
    num_workers=2,
)

## 8. Evaluate + generate heatmaps

In [ ]:
from src.evaluate import run_evaluation
from IPython.display import Image, display

CHECKPOINT = PROJECT_ROOT / 'models' / 'best_resnet50.pth'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'

results = run_evaluation(
    checkpoint_path=CHECKPOINT,
    data_root=DATA_ROOT,
    output_dir=FIGURES_DIR,
    split='test',
    generate_cams=True,
    num_cam_samples=8,
    num_failure_cases=4,
)

print(f'PR-AUC: {results["pr_auc"]:.4f}')
print(f'ROC-AUC: {results["roc_auc"]:.4f}')

In [ ]:
# Show results inline
for name in ['confusion_matrix.png', 'precision_recall_curve.png', 'training_curves.png']:
    path = FIGURES_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path), width=500))

cam_dir = FIGURES_DIR / 'cam_comparisons'
if cam_dir.exists():
    for p in sorted(cam_dir.glob('*.png'))[:3]:
        display(Image(filename=str(p), width=700))

## 9. Download results to your computer

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/results.zip'
shutil.make_archive('/content/results', 'zip', FIGURES_DIR.parent)
files.download(zip_path)
print('Downloaded reports/ and models/ are on your Drive at:', PROJECT_ROOT)

---

### Colab tips

| Issue | Fix |
|-------|-----|
| Session disconnects | Models/data on **Drive** persist; re-run from step 7 |
| CUDA out of memory | Set `batch_size=16` |
| Slow CPU training | Confirm GPU runtime is enabled |
| Free tier limits | ~12h sessions; may need Colab Pro for long runs |

All outputs are saved under `MyDrive/chest_xray_pneumonia_detection/`.